# 03.02 — PPO vision + télémétrie : apprentissage autonome sur Assetto Corsa

**Sans imitation humaine.** La politique `MultiInputPolicy` reçoit une image 1×84×84 et 9 valeurs numériques; un CNN traite les images. Géométrie CSV du circuit uniquement pour l’observation et la récompense. **L’entraînement commande réellement la voiture** uniquement après activation explicite. Commencer hors compétition, seul en piste et boîte automatique. **Stop d’urgence : interrompre le noyau, puis fermer le jeu si nécessaire.**

**Version pédales v2 :** commande accélérateur/frein exclusive selon la plus forte demande, journal enrichi (commandes envoyées, RPM, rapport, état pédales), nouveau modèle et premier essai limité à 512 pas. La synchronisation caméra/télémétrie corrigée est conservée.


In [33]:
from pathlib import Path
import sys
CWD=Path.cwd().resolve()
REPO=next((p for p in [CWD,*CWD.parents] if (p/'assetto_corsa_gym'/'assetto_corsa_gym'/'AssettoCorsaPlugin').is_dir()),None)
if REPO is None:
    raise FileNotFoundError('Ouvrir la racine IA_projet_Ouarti_ds_github ou un sous-dossier dans VS Code.')
PROJECT=REPO/'Code'
GYM_ROOT=REPO/'assetto_corsa_gym'/'assetto_corsa_gym'
assert PROJECT.is_dir(), 'Dossier Code introuvable'
if sys.version_info[:2] != (3,9):
    raise RuntimeError('Selectionner le kernel Python 3.9.13 dans VS Code. Actuel : '+sys.version)
for name in ['datasets','models','reports']:
    (PROJECT/name).mkdir(parents=True, exist_ok=True)
print('Depot :',REPO,'\nCode :',PROJECT,'\nPython :',sys.executable)

Depot : C:\Users\elkat\Documents\Github\IA_projet_Ouarti_ds_github 
Code : C:\Users\elkat\Documents\Github\IA_projet_Ouarti_ds_github\Code 
Python : c:\Users\elkat\AppData\Local\Programs\Python\Python39\python.exe


In [34]:
import socket, json, ast, time
HOST='127.0.0.1'; PORT_UDP=2345; PORT_TCP=2347

def management(command,timeout=4):
    if command not in ('get_config','get_static_info'):
        raise ValueError('Cette fonction attend une commande qui retourne des donnees.')
    with socket.create_connection((HOST,PORT_TCP),timeout=timeout) as s:
        s.settimeout(timeout)
        s.sendall(command.encode('ascii'))
        data=b''
        deadline=time.monotonic()+timeout
        while time.monotonic()<deadline and len(data)<1024*1024:
            try: chunk=s.recv(65536)
            except socket.timeout: break
            if not chunk: break
            data+=chunk
            try: return json.loads(data)
            except (ValueError,UnicodeError): pass
            try: return ast.literal_eval(data.decode('utf8'))
            except (ValueError,SyntaxError,UnicodeError): pass
    raise RuntimeError('Reponse management absente/incomplete pour '+command+': '+repr(data[:150]))

def request_reset():
    # Le protocole sensors_par NE renvoie AUCUNE reponse a la commande reset.
    with socket.create_connection((HOST,PORT_TCP),timeout=4) as s:
        s.settimeout(4)
        s.sendall(b'reset')
        # Fermer la connexion : le serveur traite la commande sans ACK.

def open_telemetry(timeout=6):
    sock=socket.socket(socket.AF_INET,socket.SOCK_DGRAM)
    sock.settimeout(.30)
    sock.connect((HOST,PORT_UDP))
    end=time.monotonic()+timeout
    try:
        while time.monotonic()<end:
            sock.send(b'connect')
            try: data=sock.recv(2**18)
            except socket.timeout: continue
            if data==b'identified': return sock
        raise TimeoutError('Le plugin UDP 2345 ne repond pas : lancer une session de conduite.')
    except Exception:
        sock.close()
        raise

def next_packet(sock,timeout=3,after_steps=None):
    deadline=time.monotonic()+timeout
    while time.monotonic()<deadline:
        try: data=sock.recv(2**18)
        except socket.timeout: continue
        if data==b'disconnect': raise ConnectionError('Connexion telemetrie interrompue')
        if data==b'identified': continue
        try: packet=json.loads(data)
        except (ValueError,UnicodeError): continue
        if not isinstance(packet,dict): continue
        if after_steps is not None and int(packet.get('steps',-1))<=int(after_steps): continue
        packet['_received_unix']=time.time()
        return packet
    raise TimeoutError('Aucun nouvel echantillon telemetrie dans le delai imparti.')

def close_telemetry(sock):
    if sock is None: return
    try: sock.send(b'disconnect')
    except OSError: pass
    sock.close()

In [35]:
import sys, time, numpy as np, cv2
PLUGIN_PATH=GYM_ROOT/'AssettoCorsaPlugin'/'plugins'/'sensors_par'
if not PLUGIN_PATH.is_dir():
    raise FileNotFoundError('Plugin Python source manquant : '+str(PLUGIN_PATH))
if str(PLUGIN_PATH) not in sys.path:
    sys.path.insert(0,str(PLUGIN_PATH))
from dual_buffer import GrabberSharedMemoryDualBuffer

class CameraReader:
    def __init__(self,config,age_limit=.55):
        if not config.get('screen_capture_enable'):
            raise RuntimeError('screen_capture_enable=False dans le plugin installe.')
        self.cam=GrabberSharedMemoryDualBuffer(int(config['final_image_width']),
                      int(config['final_image_height']),str(config['color_mode']))
        self.age_limit=float(age_limit)
        self.last_seq=-1
    def read(self,timeout=2,require_new=True):
        deadline=time.monotonic()+float(timeout)
        last_err='Aucune image stable et fraiche'
        while time.monotonic()<deadline:
            try:
                seq,stamp,arr=self.cam.get_timed_image()
                age=time.time()-float(stamp)
                if age<-.15 or age>self.age_limit:
                    last_err='Image perimee ou horloge incoherente (age %.3f s)'%age
                elif require_new and int(seq)<=self.last_seq:
                    last_err='Image non actualisee (seq %s)'%seq
                else:
                    self.last_seq=int(seq)
                    return int(seq),float(stamp),arr
            except (RuntimeError,ValueError) as e:
                last_err=str(e)
            time.sleep(.015)
        raise TimeoutError('Capture indisponible : '+last_err)

def frame_to_image84(frame):
    if frame.ndim==3:
        # La capture couleur du plugin est en BGR.
        gray=cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)
    elif frame.ndim==2:
        gray=frame
    else:
        raise ValueError('Format camera invalide : '+str(frame.shape))
    gray=cv2.resize(gray,(84,84),interpolation=cv2.INTER_AREA)
    return np.ascontiguousarray(gray[np.newaxis,:,:],dtype=np.uint8)

In [36]:
import numpy as np, pandas as pd
from scipy.spatial import cKDTree

class TrackGeometry:
    def __init__(self,path):
        df=pd.read_csv(path)
        required=['left_border_x','left_border_y','right_border_x','right_border_y']
        if not set(required).issubset(df.columns):
            raise ValueError('CSV geometrie non compatible, colonnes attendues: '+str(required))
        left=df[['left_border_x','left_border_y']].to_numpy(dtype=np.float64)
        right=df[['right_border_x','right_border_y']].to_numpy(dtype=np.float64)
        self.center=(left+right)*.5
        self.width=np.linalg.norm(left-right,axis=1)
        self.delta=np.roll(self.center,-1,axis=0)-self.center
        self.segment=np.linalg.norm(self.delta,axis=1)
        if (self.segment>.001).sum()<10:
            raise ValueError('CSV de circuit invalide: pas assez de points')
        self.arc=np.concatenate(([0.],np.cumsum(self.segment)))
        self.length=float(self.arc[-1])
        self.tree=cKDTree(self.center)
    def project(self,raw):
        xy=np.asarray([float(raw['world_position_x']),float(raw['world_position_y'])],dtype=np.float64)
        _,j=self.tree.query(xy)
        j=int(j)
        forward=self.delta[j]/max(float(self.segment[j]),1e-8)
        signed=float(forward[0]*(xy[1]-self.center[j,1])-forward[1]*(xy[0]-self.center[j,0]))
        heading=float(np.arctan2(forward[1],forward[0]))
        return float(self.arc[j]),signed,float(self.width[j]),heading

def numerical_state(raw,track):
    arc,offset,width,heading=track.project(raw)
    diff=(float(raw.get('yaw',0))-heading+np.pi)%(2*np.pi)-np.pi
    vec=[float(raw.get('speed',0))/100.,float(raw.get('accelX',0))/3.,
         float(raw.get('accelY',0))/3.,float(raw.get('angular_velocity_y',0))/4.,
         float(np.clip(offset/max(width/2.,1.),-3,3)),np.sin(diff),np.cos(diff),
         float(raw.get('accStatus',0)),float(raw.get('brakeStatus',0))]
    return np.clip(np.asarray(vec,dtype=np.float32),-5,5),arc,offset,width

def find_track_file(track_name):
    # Les fichiers .pkl sont des grilles d'occupation; ici la recompense
    # utilise les CSV bord gauche/bord droit du depot assetto_corsa_gym.
    name=str(track_name)
    path=GYM_ROOT/'AssettoCorsaConfigs'/'tracks'/(name+'.csv')
    if not path.is_file():
        raise FileNotFoundError('CSV de geometrie introuvable : '+str(path))
    return path

In [37]:
import ctypes, threading, time, numpy as np
from pathlib import Path

class VJoySafe:
    """Pilote local Windows deja valide dans 04_Test_Volant_vJoy/05_Test_Pedales."""
    X=0x30; Y=0x31; Z=0x32
    def __init__(self,device_id=1,watchdog_sec=.7):
        dll_path=Path(r'C:\Program Files\vJoy\x64\vJoyInterface.dll')
        if not dll_path.exists(): raise FileNotFoundError(str(dll_path))
        self.dll=ctypes.CDLL(str(dll_path)); self.device_id=int(device_id)
        self.dll.GetVJDStatus.argtypes=[ctypes.c_uint]
        self.dll.GetVJDStatus.restype=ctypes.c_int
        self.dll.AcquireVJD.argtypes=[ctypes.c_uint]
        self.dll.AcquireVJD.restype=ctypes.c_int
        self.dll.SetAxis.argtypes=[ctypes.c_long,ctypes.c_uint,ctypes.c_uint]
        self.dll.SetAxis.restype=ctypes.c_int
        self.dll.RelinquishVJD.argtypes=[ctypes.c_uint]
        self.dll.RelinquishVJD.restype=None
        status=self.dll.GetVJDStatus(self.device_id)
        if status!=1: raise RuntimeError('vJoy peripherique %d indisponible, statut=%d'%(self.device_id,status))
        if not self.dll.AcquireVJD(self.device_id): raise RuntimeError('Echec AcquireVJD')
        self.lock=threading.Lock(); self.active=True
        self.last_action=time.monotonic(); self.watchdog_sec=float(watchdog_sec)
        self.last_command={'steer_cmd':0.0, 'throttle_cmd':0.0, 'brake_cmd':1.0}
        self._set_axes(16384,0,32767)  # frein de securite lors de la prise de controle
        self.thread=threading.Thread(target=self._watchdog,daemon=True)
        self.thread.start()
    def _set_axes(self,x,y,z):
        with self.lock:
            for axis,value in ((self.X,x),(self.Y,y),(self.Z,z)):
                if not self.dll.SetAxis(int(value),self.device_id,axis):
                    raise RuntimeError('Echec SetAxis vJoy sur '+hex(axis))
    def action(self,a):
        a=np.asarray(a,dtype=np.float32).reshape(-1)
        if a.shape!=(3,) or not np.isfinite(a).all(): raise ValueError('Action PPO invalide')
        steer=float(np.clip(a[0],-1.,1.))
        gas=float(np.clip(a[1],0.,1.))
        brake=float(np.clip(a[2],0.,1.))
        # Politique PPO au depart aleatoire : accelerateur et frein peuvent etre positifs.
        # Une faible demande de frein ne doit plus annuler un fort accelerateur.
        # Commandes longitudinales mutuellement exclusives : retenir la plus forte.
        if gas >= brake:
            brake = 0.0
        else:
            gas = 0.0
        self._set_axes(round((steer+1)*16383.5),round(gas*32767),round(brake*32767))
        # Ces valeurs sont les commandes *effectivement envoyees* a vJoy.
        self.last_command = {'steer_cmd':steer, 'throttle_cmd':gas, 'brake_cmd':brake}
        self.last_action=time.monotonic()
    def hold(self):
        if self.active:
            self._set_axes(16384,0,32767)
            self.last_action=time.monotonic()
    def _watchdog(self):
        while self.active:
            if time.monotonic()-self.last_action>self.watchdog_sec:
                try: self.hold()
                except Exception: pass
            time.sleep(.06)
    def close(self):
        if not self.active: return
        try:self.hold()
        finally:
            self.active=False
            self.thread.join(timeout=.3)
            self.dll.RelinquishVJD(self.device_id)

In [38]:
import os, json, time, numpy as np, gymnasium as gym
from gymnasium import spaces

class LiveAssettoVisionEnv(gym.Env):
    """1 instance = 1 jeu local, 1 connexion UDP, 1 vJoy. Jamais de simulation sans jeu."""
    metadata={'render_modes':[]}
    def __init__(self,expected_track,expected_car,episode_seconds=45,
                 action_interval=.12,progress_sign=1,log_file=None):
        super().__init__()
        self.expected_track=str(expected_track)
        self.expected_car=str(expected_car)
        self.progress_sign=int(progress_sign)
        if self.progress_sign not in (-1,1): raise ValueError('progress_sign doit valoir -1 ou 1')
        self.episode_seconds=float(episode_seconds)
        self.action_interval=float(action_interval)
        self.action_space=spaces.Box(low=np.array([-1,0,0],dtype=np.float32),
                                     high=np.ones(3,dtype=np.float32),dtype=np.float32)
        self.observation_space=spaces.Dict({
            'image':spaces.Box(low=0,high=255,shape=(1,84,84),dtype=np.uint8),
            'state':spaces.Box(low=-5.,high=5.,shape=(9,),dtype=np.float32)})
        config=management('get_config'); sess=management('get_static_info')
        track=sess.get('TrackName','')
        if sess.get('TrackConfiguration'): track+='-'+str(sess['TrackConfiguration'])
        car=str(sess.get('CarName',''))
        if track!=self.expected_track or car!=self.expected_car:
            raise RuntimeError('Mauvaise session : '+track+'/'+car+' attendu '+self.expected_track+'/'+self.expected_car)
        if bool(config.get('vjoy_executed_by_server')):
            raise RuntimeError('Ce mode utilise vJoy local: laisser vjoy_executed_by_server=False.')
        if not config.get('screen_capture_enable'):
            raise RuntimeError('Capture camera desactivee dans le plugin')
        if sess.get('autoShifterOn') is False:
            raise RuntimeError('Activer la boite automatique dans Assetto Corsa avant PPO.')
        self.track=TrackGeometry(find_track_file(track))
        self.camera=CameraReader(config)
        self.sock=None; self.controller=None
        self.raw=None; self.arc=None;self.damage=0.;self.lap=0
        self.t0=0.;self.last_obs=None;self.idle_since=0.;self.episode_index=0;self.steps=0
        self.last_action=np.zeros(3,np.float32)
        self.log_file=Path(log_file) if log_file else None
        if self.log_file: self.log_file.parent.mkdir(parents=True,exist_ok=True)
        print('Config controle local: vJoy=local, plugin',config.get('sampling_freq'),'Hz, camera',config['color_mode'])
    def safe_stop(self):
        if self.controller is not None: self.controller.hold()
    def _connect(self):
        if self.controller is None:
            self.controller=VJoySafe(device_id=1)
        if self.sock is None:
            self.sock=open_telemetry()
    def _drain_pending_telemetry(self, max_packets=4096, max_seconds=0.12):
        """Vide les vieux datagrammes UDP sans bloquer ni perdre le timeout initial.

        A appeler apres le reset et avant la lecture de chaque observation :
        pendant time.sleep(), le buffer UDP conserve des mesures deja perimees.
        """
        if self.sock is None:
            return 0
        timeout_before = self.sock.gettimeout()
        removed = 0
        deadline = time.monotonic() + float(max_seconds)
        try:
            self.sock.setblocking(False)
            while removed < max_packets and time.monotonic() < deadline:
                try:
                    self.sock.recv(2**18)
                    removed += 1
                except (BlockingIOError, InterruptedError):
                    break
        finally:
            self.sock.settimeout(timeout_before)
        return removed

    def _observe(self, after_steps=None):
        """Obtient une image et une telemetrie recentes, avec delai controle.

        Si elles ne correspondent pas, on reprend une PAIRE fraiche plutot
        que de conserver la telemetrie d'un essai precedent.
        """
        deadline = time.monotonic() + 4.0
        last_error = 'Aucune paire image/telemetrie valide'
        while time.monotonic() < deadline:
            self._drain_pending_telemetry()
            remaining = deadline - time.monotonic()
            if remaining <= 0:
                break
            try:
                raw = next_packet(
                    self.sock,
                    timeout=min(0.8, remaining),
                    after_steps=after_steps,
                )
                remaining = deadline - time.monotonic()
                if remaining <= 0:
                    break
                seq, stamp, frame = self.camera.read(
                    timeout=min(0.8, remaining),
                )
            except (TimeoutError, RuntimeError, OSError, ConnectionError) as exc:
                last_error = str(exc)
                continue

            packet_stamp = float(raw.get('timestamp_unix', raw['_received_unix']))
            gap = abs(float(stamp) - packet_stamp)
            if gap > 0.40:
                now = time.time()
                last_error = (
                    'Camera/telemetrie non synchronisees: ecart %.3f s '
                    '(age image %.3f s, age telemetrie %.3f s)'
                    % (gap, now - float(stamp), now - packet_stamp)
                )
                continue

            state, arc, lateral, width = numerical_state(raw, self.track)
            obs = {
                'image': frame_to_image84(frame),
                'state': state,
            }
            return raw, obs, arc, lateral, width, seq, gap
        raise TimeoutError('Impossible de synchroniser les capteurs: ' + last_error)
    def reset(self,*,seed=None,options=None):
        super().reset(seed=seed)
        self._connect()
        self.safe_stop()
        # L'API reset n'accuse pas reception. Le code doit attendre de nouvelles mesures.
        request_reset()
        time.sleep(0.6)  # laisser au simulateur le temps de reinitialiser la voiture
        self._drain_pending_telemetry()  # supprimer les mesures datant d'avant le reset
        old_steps=None
        last_err=None
        for _ in range(3):
            try:
                raw,o,arc,lat,width,seq,gap=self._observe(after_steps=old_steps)
                break
            except (TimeoutError,RuntimeError,OSError,ConnectionError) as e:
                last_err=e;time.sleep(.12)
        else:
            self.safe_stop()
            raise RuntimeError('Impossible de recuperer observation apres reset: '+str(last_err))
        self.raw=raw;self.last_obs=o;self.arc=arc
        self.damage=float(raw.get('damage_total',0))
        self.lap=int(raw.get('LapCount',0))
        self.idle_since=time.monotonic()
        self.t0=time.monotonic();self.steps=0;self.episode_index+=1
        self.last_action=np.zeros(3,np.float32)
        return o,{'camera_gap_s':gap,'camera_sequence':seq,'track':self.expected_track}
    def step(self,action):
        if self.sock is None or self.raw is None:
            raise RuntimeError('reset() doit etre appele avant step()')
        action=np.asarray(action,dtype=np.float32)
        try:
            self.controller.action(action)
            time.sleep(self.action_interval)
            raw,o,arc,lateral,width,seq,gap=self._observe(after_steps=self.raw.get('steps'))
            dt=float(raw.get('timestamp_unix',raw['_received_unix']))-float(self.raw.get('timestamp_unix',self.raw['_received_unix']))
            if not .005<dt<1.0: raise RuntimeError('Delta temps telemetrie invalide: %s'%dt)
            move=(arc-self.arc+self.track.length/2)%self.track.length-self.track.length/2
            move*=self.progress_sign
            if abs(move)>max(4.,100.*dt*1.7):
                # Collision, teleportation, mauvaise projection ou horodatage : episode invalide.
                raise RuntimeError('Saut de position non physique (%.1f m sur %.3f s)'%(move,dt))
            tyres=int(raw.get('numberOfTyresOut',0))
            offtrack=bool(tyres>2 or abs(lateral)>width*.5)
            new_damage=float(raw.get('damage_total',0))
            collision=bool(new_damage>self.damage+.02)
            new_lap=int(raw.get('LapCount',0))
            lap_finished=new_lap>self.lap
            now=time.monotonic(); speed=float(raw.get('speed',0))
            if speed>1.:self.idle_since=now
            idle=bool(now-self.idle_since>6.)
            wrong_way=bool(move<-.25)
            # Recompense sans trajectoire humaine: progression geometrique + penalites.
            reward=float(np.clip(move,-5.,8.) -.02 - .015*abs(float(o['state'][4]))
                         -.008*abs(float(action[0])-float(self.last_action[0]))
                         -25.*offtrack-35.*collision-8.*idle+70.*lap_finished)
            terminated=bool(offtrack or collision or idle)
            truncated=bool(now-self.t0>=self.episode_seconds)
            self.last_obs=o;self.raw=raw;self.arc=arc;self.damage=new_damage;self.lap=new_lap
            self.last_action=action.copy();self.steps+=1
            # Enregistrer les commandes reelles et les retours des pedales.
            # Utile pour distinguer exploration PPO et probleme de configuration du jeu.
            sent=dict(self.controller.last_command)
            def telemetry_number(*keys):
                for key in keys:
                    if key in raw and raw[key] is not None:
                        try: return float(raw[key])
                        except (ValueError,TypeError): pass
                return None
            info={'progress_m':float(move),'speed_ms':speed,'offtrack':offtrack,'collision':collision,
                  'lap_finished':lap_finished,'idle':idle,'wrong_way':wrong_way,'camera_gap_s':gap,
                  'camera_sequence':seq,'episode_id':self.episode_index,
                  **sent,
                  'accStatus':telemetry_number('accStatus'),
                  'brakeStatus':telemetry_number('brakeStatus'),
                  'rpm':telemetry_number('rpm','RPM','engineRPM'),
                  'gear':telemetry_number('gear','Gear','currentGear','CurrentGear')}
            if terminated or truncated: self.safe_stop()
            if self.log_file:
                with self.log_file.open('a',encoding='utf8') as f:
                    f.write(json.dumps({'time':time.time(),'reward':reward,'terminated':terminated,
                             'truncated':truncated,**info},ensure_ascii=False)+'\n')
            return o,reward,terminated,truncated,info
        except (OSError,RuntimeError,TimeoutError,ConnectionError,ValueError) as exc:
            self.safe_stop()
            if self.last_obs is None:raise
            return self.last_obs,-40.,False,True,{'sensor_error':str(exc),'episode_id':self.episode_index}
    def close(self):
        self.safe_stop()
        if self.sock is not None:
            close_telemetry(self.sock);self.sock=None
        if self.controller is not None:
            self.controller.close();self.controller=None

## Paramètres à vérifier avant de lancer

- La voiture et le circuit doivent correspondre exactement à la session visible dans `get_static_info`.
- Le plugin réellement installé conserve **`vjoy_executed_by_server=False`**, `sampling_freq=50`, `ego_sampling_freq=25`, caméra activée : ces valeurs déjà validées sont compatibles avec ce **mode local vJoy**.
- Le port UDP 2345 n'accepte qu'un client actif; fermer les autres notebooks de télémétrie.
- La géométrie utilisée est le CSV `ks_barcelona-layout_gp.csv`, pas un `.pkl` ni une trajectoire humaine.
- Ne pas activer avant d'avoir lu les vérifications de la cellule suivante.
- **Pédales v2 :** lorsque PPO demande accélérateur ET frein, la commande la plus forte l'emporte et l'autre est mise à zéro. Les commandes effectives et les retours télémétrie sont ajoutés au journal JSONL.
- **Nouvelle expérience :** le modèle et les journaux utilisent un autre nom, afin de ne pas mélanger les résultats avec les épisodes précédents.
- **Premier test :** 512 pas, puis vérifier les commandes de pédales, la vitesse maximale et la progression avant un entraînement plus long.


In [39]:
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback, CallbackList, CheckpointCallback
from stable_baselines3.common.monitor import Monitor
import json, time, torch

TRACK_ATTENDU='ks_barcelona-layout_gp'
VOITURE_ATTENDUE='abarth500'
DIRECTION_PROGRESSION=1  # a verifier : +1 si l'arc du CSV avance dans le sens normal; -1 sinon
DUREE_EPISODE_S=45
INTERVALLE_ACTION_S=.12
TOTAL_TIMESTEPS=512  # essai de diagnostic; augmenter apres validation de la propulsion
ACTIVER_ENTRAINEMENT_REEL=True  # OPT-IN, aucun mouvement si False
MODELE_NOM='ppo_vision_telemetrie_pedales_v2'

class PauseDuringUpdates(BaseCallback):
    def __init__(self,core):
        super().__init__();self.core=core
    def _on_step(self):return True
    def _on_rollout_end(self):self.core.safe_stop()

config_live=None;static_live=None
if ACTIVER_ENTRAINEMENT_REEL:
    config_live=management('get_config')
    static_live=management('get_static_info')
    print('Verification du jeu:',static_live.get('TrackName'),static_live.get('TrackConfiguration'),static_live.get('CarName'))
    print('Commande locale:',config_live.get('vjoy_executed_by_server'))
else:
    print('MODE SECURISE: aucun entrainement, aucune commande. Mettre True pour debuter intentionnellement.')

Verification du jeu: ks_barcelona layout_gp abarth500
Commande locale: False


In [40]:
if ACTIVER_ENTRAINEMENT_REEL:
    core=None
    prefix=PROJECT/'models'/MODELE_NOM
    log_dir=PROJECT/'reports'/'ppo_vision_pedales_v2'
    log_dir.mkdir(parents=True,exist_ok=True)
    try:
        core=LiveAssettoVisionEnv(TRACK_ATTENDU,VOITURE_ATTENDUE,
             episode_seconds=DUREE_EPISODE_S,action_interval=INTERVALLE_ACTION_S,
             progress_sign=DIRECTION_PROGRESSION,log_file=log_dir/'ppo_steps.jsonl')
        env=Monitor(core,filename=str(log_dir/'monitor.csv'))
        agent=PPO('MultiInputPolicy',env,n_steps=128,batch_size=32,n_epochs=4,
                  learning_rate=3e-4,gamma=.99,gae_lambda=.95,ent_coef=.005,
                  verbose=1,device='auto',tensorboard_log=str(log_dir/'tensorboard'))
        callbacks=CallbackList([PauseDuringUpdates(core),
            CheckpointCallback(save_freq=1024,save_path=str(PROJECT/'models'/'checkpoints'),
                               name_prefix=MODELE_NOM,save_replay_buffer=False)])
        agent.learn(total_timesteps=int(TOTAL_TIMESTEPS),callback=callbacks,
                    tb_log_name=MODELE_NOM,progress_bar=False)
        core.safe_stop()
        agent.save(str(prefix))
        meta={'model':MODELE_NOM,'track':TRACK_ATTENDU,'car':VOITURE_ATTENDUE,
              'progress_sign':DIRECTION_PROGRESSION,'control':'vjoy_local_xyz_pedales_exclusives_v2',
              'observation':'dict_image_1x84x84_state_9','image_color':'gray_from_plugin',
              'total_timesteps':int(agent.num_timesteps),'saved_at':time.time()}
        prefix.with_name(prefix.name+'_meta.json').write_text(json.dumps(meta,indent=2),encoding='utf8')
        print('Modele:',prefix.with_suffix('.zip'),'metadata:',prefix.with_name(prefix.name+'_meta.json'))
    except KeyboardInterrupt:
        print('Interruption manuelle : frein de securite demande.')
        if core is not None: core.safe_stop()
        raise
    finally:
        if core is not None: core.close()

Config controle local: vJoy=local, plugin 50 Hz, camera gray
Using cpu device
Wrapping the env in a DummyVecEnv.
Logging to C:\Users\elkat\Documents\Github\IA_projet_Ouarti_ds_github\Code\reports\ppo_vision_pedales_v2\tensorboard\ppo_vision_telemetrie_pedales_v2_1
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 39.3     |
|    ep_rew_mean     | -18.5    |
| time/              |          |
|    fps             | 3        |
|    iterations      | 1        |
|    time_elapsed    | 36       |
|    total_timesteps | 128      |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 35.4        |
|    ep_rew_mean          | -17.7       |
| time/                   |             |
|    fps                  | 4           |
|    iterations           | 2           |
|    time_elapsed         | 59          |
|    total_timesteps      | 256         |
| train/         

In [41]:
# Diagnostic facultatif apres un premier essai : lire les journaux sans commander la voiture.
import json
from pathlib import Path
from collections import defaultdict

journal = PROJECT / 'reports' / 'ppo_vision_pedales_v2' / 'ppo_steps.jsonl'
if not journal.exists():
    print('Aucun journal v2 pour le moment. Lancer volontairement un essai PPO court.')
else:
    episodes = defaultdict(list)
    with journal.open('r', encoding='utf8') as f:
        for line in f:
            try:
                rec = json.loads(line)
                episodes[rec['episode_id']].append(rec)
            except (ValueError, KeyError):
                continue
    print('10 derniers episodes :')
    print('ep | pas | reward | progression(m) | vitesse max(km/h) | gaz moy | frein moy')
    for ep in sorted(episodes)[-10:]:
        steps = episodes[ep]
        n = len(steps)
        mean = lambda key: sum(float(p.get(key) or 0) for p in steps) / n
        print(f"{ep:>2} | {n:>3} | {sum(float(p.get('reward') or 0) for p in steps):>7.2f} | "
              f"{sum(float(p.get('progress_m') or 0) for p in steps):>14.2f} | "
              f"{max(float(p.get('speed_ms') or 0) for p in steps)*3.6:>18.1f} | "
              f"{mean('throttle_cmd'):>7.2f} | {mean('brake_cmd'):>9.2f}")


10 derniers episodes :
ep | pas | reward | progression(m) | vitesse max(km/h) | gaz moy | frein moy
 2 |  63 |   -5.59 |           4.61 |                8.8 |    0.24 |      0.25
 3 |  39 |   -9.37 |           0.00 |                2.5 |    0.23 |      0.21
 4 |  10 |   -0.33 |           0.00 |                0.3 |    0.31 |      0.31
 5 |  40 |   -9.43 |           0.00 |                2.8 |    0.20 |      0.24
 6 |  40 |   -9.37 |           0.00 |                2.7 |    0.15 |      0.33
 7 |  39 |   -9.35 |           0.00 |                3.3 |    0.23 |      0.30
 8 | 123 |  -21.94 |           7.69 |                6.4 |    0.29 |      0.26
 9 |  31 |   -9.09 |           0.00 |                1.4 |    0.16 |      0.30
10 |  79 |  -18.67 |           9.16 |               11.3 |    0.32 |      0.16
11 |  31 |    0.39 |           1.51 |               10.6 |    0.42 |      0.23


**Ce que fait le réseau de vision** : le CNN apprend automatiquement des caractéristiques visuelles utiles à maximiser les récompenses. Ce n'est **pas** une U-Net de segmentation sémantique. Produire des masques « herbe, vibreur, asphalte » exige une source de labels ou une autre méthode explicitement implémentée.

**Difficulté expérimentale** : l'entraînement de zéro sur le simulateur en temps réel est coûteux, la progression peut rester faible au départ, et les transitions avec problème de capture sont tronquées plutôt que d'inventer une observation. Les resets doivent être vérifiés sur ton installation réelle.